In [1]:
from sklearn.model_selection import train_test_split, cross_val_predict, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from plotly.subplots import make_subplots
import plotly.graph_objs as go
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import RandomizedSearchCV
import shap
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
import plotly.express as px
from catboost import CatBoostRegressor
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from sklearn.compose import TransformedTargetRegressor

In [2]:
RANDOM_STATE = 12345

In [3]:
df = pd.read_csv("../data/diamonds.csv")

In [4]:
df = df.drop_duplicates()

In [5]:
x_precentile_99 = df['x'].quantile(0.99)
y_precentile_99 = df['y'].quantile(0.99)
z_precentile_99 = df['z'].quantile(0.99)
x_precentile_01 = df['x'].quantile(0.01)
y_precentile_01 = df['y'].quantile(0.01)
z_precentile_01 = df['z'].quantile(0.01)
df = df[df['x']<= x_precentile_99]
df = df[df['y']<= y_precentile_99]
df = df[df['z']<= z_precentile_99]
df = df[df['x']>= x_precentile_01]
df = df[df['y']>= y_precentile_01]
df = df[df['z']>= z_precentile_01]

In [6]:
X = df.drop('price', axis = 1)

In [7]:
y = df['price']

In [8]:
X = X.drop(columns = 'Unnamed: 0')

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = RANDOM_STATE)

In [10]:
X_train['volume'] = X_train['x'] * X_train['y'] * X_train['z']

# 2. Карат на объём
X_train['carat_per_volume'] = X_train['carat'] / (X_train['volume'] + 1e-8)

# 3. Отношение table к depth
X_train['table_depth_ratio'] = X_train['table'] / (X_train['depth'] + 1e-8)


In [11]:
num_cols = X_train.select_dtypes(include='number').columns.tolist()

In [12]:
cat_cols = X_train.select_dtypes(include='object').columns.tolist()

/var/folders/jt/pfrf2kqn59d1ph0418pyq4mr0000gn/T/ipykernel_48596/4180746908.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include='object').columns.tolist()


In [13]:
cat_cols = [col for col in cat_cols if col not in ['cut', 'color', 'clarity']]

In [14]:
ord_cols = ['cut', 'color', 'clarity']

In [15]:
cut_order = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
color_order = ['J', 'I', 'H', 'G', 'F', 'E', 'D']
clarity_order = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

In [16]:
preprocessor = ColumnTransformer([
    ('num', 'passthrough', num_cols),

    ('ord',
     OrdinalEncoder(
         categories=[
             cut_order,
             color_order,
             clarity_order
         ]
     ),
     ord_cols),

    ('cat',
     OneHotEncoder(handle_unknown='ignore'),
     cat_cols)
])

In [17]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [18]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", CatBoostRegressor(
        iterations=300,
        depth=6,
        learning_rate=0.1,
        verbose=0,
        random_state=RANDOM_STATE
    ))
])

In [21]:
param_distributions = {
    "model__iterations": [300, 500, 700],         
    "model__depth": [6, 8, 10],                  
    "model__learning_rate": [0.05, 0.1, 0.15],      
    "model__l2_leaf_reg": [1, 3, 5],             
    "model__random_strength": [1, 3]                
}

In [22]:
random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step... verbose=0))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__depth': [6, 8, ...], 'model__iterations': [300, 500, ...], 'model__l2_leaf_reg': [1, 3, ...], 'model__learning_rate': [0.05, 0.1, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",12345
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score in

In [ ]:
random_search.best_params_
-random_search.best_score_

np.float64(554.0693206823577)

In [ ]:
best_model = random_search.best_estimator_

y_pred = best_model.predict(X_test)

final_metrics = {
    "mae": mean_absolute_error(y_test, y_pred),
    "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
    "r2": r2_score(y_test, y_pred)
}

pd.DataFrame([final_metrics])

,mae,rmse,r2
0,265.497546,535.39528,0.981633


Лучшая модель CatBoostRegressor с параметрами:depth=10, iterations=700, l2_leaf_reg=1, learning_rate=0.05, loss_function='RMSE', random_state=12345, random_strength=1, verbose=0

